In [ ]:
from neo4j import GraphDatabase

# Configurações do Neo4j
NEO4J_URI = "bolt://localhost:7687"  # Substitua pelo URI do seu banco Neo4j
NEO4J_USER = "neo4j"  # Usuário padrão do Neo4j
NEO4J_PASSWORD = "password"  # Substitua pela senha do seu banco

class ProductRecommendationSystem:

    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def create_schema(self):
        """
        Cria um esquema inicial para produtos, usuários e relações de compra.
        """
        with self.driver.session() as session:
            session.run("""
            CREATE CONSTRAINT FOR (p:Product) REQUIRE p.id IS UNIQUE;
            CREATE CONSTRAINT FOR (u:User) REQUIRE u.id IS UNIQUE;
            """)

    def insert_sample_data(self):
        """
        Insere dados de exemplo no banco de dados.
        """
        with self.driver.session() as session:
            session.run("""
            CREATE (p1:Product {id: '1', name: 'Smartphone', category: 'Electronics'})
            CREATE (p2:Product {id: '2', name: 'Laptop', category: 'Electronics'})
            CREATE (p3:Product {id: '3', name: 'Headphones', category: 'Accessories'})
            CREATE (p4:Product {id: '4', name: 'Camera', category: 'Electronics'})
            CREATE (u1:User {id: '101', name: 'Alice'})
            CREATE (u2:User {id: '102', name: 'Bob'})
            CREATE (u1)-[:PURCHASED]->(p1)
            CREATE (u1)-[:PURCHASED]->(p2)
            CREATE (u2)-[:PURCHASED]->(p2)
            CREATE (u2)-[:PURCHASED]->(p3)
            """)

    def recommend_products(self, user_id):
        """
        Recomendação baseada em produtos comprados por outros usuários com interesses semelhantes.
        """
        with self.driver.session() as session:
            query = """
            MATCH (u:User {id: $user_id})-[:PURCHASED]->(p:Product)<-[:PURCHASED]-(other:User)-[:PURCHASED]->(rec:Product)
            WHERE NOT (u)-[:PURCHASED]->(rec)
            RETURN DISTINCT rec.name AS recommended_product, COUNT(*) AS score
            ORDER BY score DESC
            LIMIT 5
            """
            result = session.run(query, user_id=user_id)
            return [{"product": record["recommended_product"], "score": record["score"]} for record in result]

# Inicializando o sistema
if __name__ == "__main__":
    # Conexão com o Neo4j
    recommender = ProductRecommendationSystem(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

    # Configuração inicial
    print("Criando esquema e inserindo dados de exemplo...")
    recommender.create_schema()
    recommender.insert_sample_data()

    # Exemplo de recomendação
    user_id = "101"  # Usuário para quem gerar recomendações
    recommendations = recommender.recommend_products(user_id)
    print(f"Recomendações para o usuário {user_id}:")
    for rec in recommendations:
        print(f"Produto: {rec['product']}, Score: {rec['score']}")

    # Fechando conexão
    recommender.close()
